In [ ]:
import numpy as np
from typing import List, Dict, Tuple

# ----------------------------------------------------------------------
# 1. VALIDATION HELPERS
# ----------------------------------------------------------------------

def _validate_dim_vector(vec, name="dimension vector"):
    # FIX: enforce float dtype explicitly so integer-typed inputs don't cause
    # integer arithmetic in matrix products downstream.
    vec = np.asarray(vec, dtype=float)
    if vec.shape != (4,):
        raise ValueError(f"{name} must have shape (4,), got {vec.shape}")
    if not np.all(np.isfinite(vec)):
        raise ValueError(f"{name} contains non-finite entries: {vec}")
    return vec

def _validate_positive(value, name="value"):
    # Retained for constants that MUST be positive (reference magnitudes used as
    # denominators / log arguments in build_system).
    value = float(value)
    if not np.isfinite(value):
        raise ValueError(f"{name} must be finite, got {value}")
    if value <= 0:
        raise ValueError(f"{name} must be positive, got {value}")
    return value

def _validate_nonzero(value, name="value"):
    # FIX: general physical quantities may be negative (velocity components,
    # charges, signed energies). Scaling acts on MAGNITUDE; sign is preserved.
    # We only forbid exactly zero because log|0| is undefined; a zero-magnitude
    # quantity is scale-invariant anyway and should be handled separately.
    value = float(value)
    if not np.isfinite(value):
        raise ValueError(f"{name} must be finite, got {value}")
    if value == 0.0:
        raise ValueError(
            f"{name} is exactly zero; magnitude scaling via log is undefined. "
            f"A zero-valued quantity is invariant under unit rescaling."
        )
    return value

def _split_mag_sign(value):
    # FIX: separate magnitude and sign so log-based scaling works for signed values.
    return abs(value), np.sign(value)

# ----------------------------------------------------------------------
# 2. SYSTEM BUILDER
# ----------------------------------------------------------------------

def build_system(constraints: List[Dict],
                 cond_threshold: float = 1e12) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build passive transformation matrix B and active log-scales x from 4 constraints.

    Each constraint dict must have:
        'target'   : float > 0  (desired numerical value in the new system)
        'si_value' : float > 0  (numerical value in SI)
        'si_dims'  : array_like, length 4  (SI dimension vector: [M, L, T, Θ])

    Mathematical note
    -----------------
    A has the SI dimension vectors of the constraints as its COLUMNS.
    B = A^{-1} is the passive change-of-basis: d_new = B @ d_si maps an SI
    dimension vector into the constraint basis, where the i-th coordinate counts
    how many factors of constraint-constant i a quantity carries.

    The log-scales x satisfy, for each constraint k (whose d_new = e_k):
        log(new_value_k) = log(si_value_k) + e_k . x = log(si_value_k) + x_k
    Setting new_value_k = target_k gives x_k = log(target_k / si_value_k).
    """
    if len(constraints) != 4:
        raise ValueError(f"Exactly 4 constraints required, got {len(constraints)}")

    A_cols = []
    targets = []
    si_values = []
    for i, c in enumerate(constraints):
        si_dims = _validate_dim_vector(c['si_dims'], f"constraint[{i}].si_dims")
        sv = _validate_positive(c['si_value'], f"constraint[{i}].si_value")
        tg = _validate_positive(c['target'],   f"constraint[{i}].target")
        A_cols.append(si_dims)
        si_values.append(sv)
        targets.append(tg)

    A = np.column_stack(A_cols)

    # FIX: replace the scale-dependent np.isclose(det, 0) test with a condition
    # number check. det can be tiny for a well-conditioned matrix (or large for
    # an ill-conditioned one) purely due to scaling, so det is a poor singularity
    # diagnostic. cond directly measures invertibility/numerical stability.
    cond = np.linalg.cond(A)
    if not np.isfinite(cond) or cond > cond_threshold:
        raise ValueError(
            f"Constraint dimension vectors are linearly dependent or "
            f"ill-conditioned (cond(A) = {cond:.3e} > {cond_threshold:.1e})."
        )

    B = np.linalg.inv(A)
    # x_k = log(target_k / si_value_k); both validated positive above.
    x = np.log(np.asarray(targets) / np.asarray(si_values))

    return B, x

# ----------------------------------------------------------------------
# 3. PASSIVE TRANSFORMATIONS (coordinate change only, values unchanged)
# ----------------------------------------------------------------------
#
# WARNING (physics interpretation): 'new_dims' produced here are COORDINATES of
# the dimension vector in the constraint basis, NOT physical [M,L,T,Θ] exponents.
# They are meaningful only as input to the active step. Do not interpret them as
# SI-style dimensional exponents.

def passive_apply(quantity: Dict, B: np.ndarray) -> Dict:
    """
    Apply passive change-of-basis to a single quantity.
    Numerical value is unchanged; the dimension vector is expressed in the new basis.
    """
    name = quantity.get('name', 'unnamed')
    # FIX: allow signed values; only magnitude is ever scaled (and not here at all).
    si_value = _validate_nonzero(quantity['value'], f"{name}.value")
    si_dims = _validate_dim_vector(quantity['si_dims'], f"{name}.si_dims")

    B = np.asarray(B, dtype=float)
    if B.shape != (4, 4):
        raise ValueError("B must be a 4x4 matrix")

    d_new = B @ si_dims

    return {
        'name': name,
        'value': si_value,
        'si_value': si_value,
        'si_dims': si_dims.copy(),
        'new_value': si_value,         # unchanged in passive step
        'new_dims': d_new              # basis coordinates, NOT SI exponents
    }

def passive_apply_bulk(quantities: List[Dict], B: np.ndarray) -> List[Dict]:
    """Vectorised passive transformation for many quantities."""
    if not quantities:
        return []

    B = np.asarray(B, dtype=float)
    if B.shape != (4, 4):
        raise ValueError("B must be a 4x4 matrix")

    names, si_values, si_dims_list = [], [], []
    for q in quantities:
        nm = q.get('name', 'unnamed')
        names.append(nm)
        si_values.append(_validate_nonzero(q['value'], f"{nm}.value"))  # FIX: signed ok
        si_dims_list.append(_validate_dim_vector(q['si_dims'], f"{nm}.si_dims"))

    D_si = np.vstack(si_dims_list)
    D_new = D_si @ B.T   # row i: B @ si_dims_i

    result = []
    for i in range(len(quantities)):
        result.append({
            'name': names[i],
            'value': si_values[i],
            'si_value': si_values[i],
            'si_dims': si_dims_list[i].copy(),
            'new_value': si_values[i],
            'new_dims': D_new[i].copy()
        })
    return result

# ----------------------------------------------------------------------
# 4. ACTIVE TRANSFORMATIONS (value rescaling only, given new-basis dims)
# ----------------------------------------------------------------------

def active_apply(quantity: Dict, d_new: np.ndarray, x: np.ndarray) -> Dict:
    """
    Apply active scaling to a single quantity.

    Requires the dimension vector in the constraint basis (d_new) and log-scales x.
    Scaling acts on the magnitude:  new_value = sign(v) * |v| * exp(d_new . x).
    """
    name = quantity.get('name', 'unnamed')
    si_value = _validate_nonzero(quantity['value'], f"{name}.value")  # FIX: signed ok
    si_dims = _validate_dim_vector(quantity['si_dims'], f"{name}.si_dims")
    d_new = _validate_dim_vector(d_new, f"{name}.d_new")

    x = np.asarray(x, dtype=float)
    if x.shape != (4,):
        raise ValueError("x must be a length-4 vector")
    if not np.all(np.isfinite(x)):
        raise ValueError("x contains non-finite entries")

    # FIX: operate on magnitude, restore sign, so negative quantities are valid.
    mag, sgn = _split_mag_sign(si_value)
    log_new = np.log(mag) + float(np.dot(d_new, x))
    new_value = sgn * np.exp(log_new)

    return {
        'name': name,
        'value': si_value,
        'si_value': si_value,
        'si_dims': si_dims.copy(),
        'new_value': new_value,
        'new_dims': d_new.copy()
    }

def active_apply_bulk(quantities: List[Dict], D_new: np.ndarray, x: np.ndarray) -> List[Dict]:
    """Vectorised active scaling for many quantities."""
    if not quantities:
        return []

    D_new = np.asarray(D_new, dtype=float)
    if D_new.ndim != 2 or D_new.shape[1] != 4:
        raise ValueError("D_new must be a 2D array with 4 columns")
    if D_new.shape[0] != len(quantities):
        raise ValueError("Rows in D_new must match number of quantities")
    if not np.all(np.isfinite(D_new)):
        raise ValueError("D_new contains non-finite entries")

    x = np.asarray(x, dtype=float)
    if x.shape != (4,):
        raise ValueError("x must be a length-4 vector")
    if not np.all(np.isfinite(x)):
        raise ValueError("x contains non-finite entries")

    names, si_values, si_dims_list = [], [], []
    for q in quantities:
        nm = q.get('name', 'unnamed')
        names.append(nm)
        si_values.append(_validate_nonzero(q['value'], f"{nm}.value"))  # FIX: signed ok
        si_dims_list.append(_validate_dim_vector(q['si_dims'], f"{nm}.si_dims"))

    # FIX: split magnitude/sign vectorially; cast to ndarray for safe log.
    si_arr = np.asarray(si_values, dtype=float)
    mags = np.abs(si_arr)
    signs = np.sign(si_arr)

    log_si = np.log(mags)
    log_new = log_si + D_new @ x
    new_values = signs * np.exp(log_new)

    result = []
    for i in range(len(quantities)):
        result.append({
            'name': names[i],
            'value': si_values[i],
            'si_value': si_values[i],
            'si_dims': si_dims_list[i].copy(),
            'new_value': float(new_values[i]),
            'new_dims': D_new[i].copy()
        })
    return result

# ----------------------------------------------------------------------
# 5. FULL TRANSFORMATIONS (passive + active combined)
# ----------------------------------------------------------------------

def full_apply(quantity: Dict, B: np.ndarray, x: np.ndarray) -> Dict:
    """Apply passive then active transformation to a single quantity."""
    passive_out = passive_apply(quantity, B)
    active_out = active_apply(passive_out, passive_out['new_dims'], x)
    return active_out

def full_apply_bulk(quantities: List[Dict], B: np.ndarray, x: np.ndarray) -> List[Dict]:
    """Vectorised full transformation for many quantities."""
    if not quantities:
        return []

    passive_results = passive_apply_bulk(quantities, B)
    D_new = np.vstack([r['new_dims'] for r in passive_results])
    active_results = active_apply_bulk(passive_results, D_new, x)
    return active_results

# ----------------------------------------------------------------------
# 6. CONVENIENCE EXTRACTOR
# ----------------------------------------------------------------------

def extract_new_values(transformed: List[Dict]) -> List[float]:
    """Extract just the new numerical values from a list of transformed dicts."""
    return [d['new_value'] for d in transformed]

# ----------------------------------------------------------------------
# 7. EXAMPLE USAGE
# ----------------------------------------------------------------------

if __name__ == "__main__":

    # ---- Define constraints: set c = ħ = G = k_B = 1 ----
    constraints = [
        {'target': 1.0, 'si_value': 299792458.0,
         'si_dims': np.array([0.0, 1.0, -1.0, 0.0])},   # c   [m/s]
        {'target': 1.0, 'si_value': 1.054571817e-34,
         'si_dims': np.array([1.0, 2.0, -1.0, 0.0])},   # ħ   [kg m^2 / s]
        {'target': 1.0, 'si_value': 6.67430e-11,
         'si_dims': np.array([-1.0, 3.0, -2.0, 0.0])},  # G   [m^3 / (kg s^2)]
        {'target': 1.0, 'si_value': 1.380649e-23,
         'si_dims': np.array([1.0, 2.0, -2.0, -1.0])},  # k_B [kg m^2 / (s^2 K)]
    ]

    B, x = build_system(constraints)

    print("Passive transformation matrix B (SI -> Planck/constraint basis):")
    print(B)
    print("\nActive log-scales x:")
    print(x)
    print("\nActive scaling factors exp(x):")
    print(np.exp(x))

    quantities = [
        {'name': 'proton_mass',              'value': 1.67262192e-27,
         'si_dims': np.array([1.0, 0.0, 0.0, 0.0])},
        {'name': 'electron_mass',            'value': 9.1093837e-31,
         'si_dims': np.array([1.0, 0.0, 0.0, 0.0])},
        {'name': 'planck_length',            'value': 1.616255e-35,
         'si_dims': np.array([0.0, 1.0, 0.0, 0.0])},
        {'name': 'planck_time',              'value': 5.391247e-44,
         'si_dims': np.array([0.0, 0.0, 1.0, 0.0])},
        {'name': 'planck_temperature',       'value': 1.416784e+32,
         'si_dims': np.array([0.0, 0.0, 0.0, 1.0])},
        {'name': 'fine_structure_constant',  'value': 7.29735256e-3,
         'si_dims': np.array([0.0, 0.0, 0.0, 0.0])},
        {'name': 'gravitational_constant',   'value': 6.67430e-11,
         'si_dims': np.array([-1.0, 3.0, -2.0, 0.0])},
        # FIX: demonstrate a signed quantity now works (e.g. an x-velocity component).
        {'name': 'neg_velocity_component',   'value': -1.0e6,
         'si_dims': np.array([0.0, 1.0, -1.0, 0.0])},
    ]

    transformed = full_apply_bulk(quantities, B, x)

    print("\n" + "=" * 78)
    print("TRANSFORMED QUANTITIES IN PLANCK UNITS (c = h-bar = G = k_B = 1)")
    print("=" * 78)
    print(f"{'Name':<26} {'SI Value':>14} {'Planck Value':>18}   Basis coords")
    print("-" * 78)
    for t in transformed:
        print(f"{t['name']:<26} {t['si_value']:>14.6e} {t['new_value']:>18.8e}   {t['new_dims']}")

    new_vals = extract_new_values(transformed)
    print("\nJust the Planck-unit numerical values:")
    print(new_vals)

    print("\n" + "=" * 78)
    print("VERIFICATION: constrained constants should equal 1")
    print("=" * 78)
    check_constraints = [
        {'name': 'c',     'value': 299792458.0,    'si_dims': np.array([0.0, 1.0, -1.0, 0.0])},
        {'name': 'h-bar', 'value': 1.054571817e-34,'si_dims': np.array([1.0, 2.0, -1.0, 0.0])},
        {'name': 'G',     'value': 6.67430e-11,    'si_dims': np.array([-1.0, 3.0, -2.0, 0.0])},
        {'name': 'k_B',   'value': 1.380649e-23,   'si_dims': np.array([1.0, 2.0, -2.0, -1.0])},
    ]
    verified = full_apply_bulk(check_constraints, B, x)
    for v in verified:
        print(f"{v['name']:<6}: {v['new_value']:.15f}")

    # FIX: quantitative self-check rather than visual inspection only.
    max_err = max(abs(v['new_value'] - 1.0) for v in verified)
    print(f"\nMax deviation from 1.0 across constraints: {max_err:.3e}")
    assert max_err < 1e-9, "Constraint verification failed beyond tolerance."
    print("Constraint verification PASSED.")
